# Sat Query — M3 Optical + SAR Multimodal Analysis Experiments

This notebook demonstrates the end-to-end processing pipeline for **Module 3 (M3) Optical + SAR** within the **Sat Query** system.

### Complete Workflow Overview:
1. **Imports & Setup**
2. **Configuration**
3. **Load Optical Satellite Image (Sentinel-2)**
4. **Visualize Optical Imagery (True Color RGB & False Color CIR)**
5. **Optical Preprocessing (Cloud Masking & Normalization)**
6. **Calculate NDVI & Spectral Indices**
7. **Load SAR Satellite Image (Sentinel-1)**
8. **Visualize SAR Polarizations (VV & VH)**
9. **SAR Preprocessing (Calibration, Lee Filter, Terrain Adapter, Normalization)**
10. **Geospatial Reprojection (SAR to Optical Grid)**
11. **Cross-Modal Fine Alignment (Phase Correlation)**
12. **Registration Quality Validation (Overlap, NMI, Structural Correlation)**
13. **Early Multimodal Fusion (6-Channel Composite)**
14. **Multimodal Deep Learning Inference (Two-Stream CNN)**
15. **Multimodal Confidence Estimation**
16. **Final Multimodal Visualizations**
17. **Summary & Module Handoff Interfaces**

## 1. Imports & Environment Setup

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.crs import CRS
import affine

# Ensure project root is on sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from modules.optical_sar import (
    OpticalSARConfig,
    run_optical_sar_pipeline,
    load_optical,
    load_sar,
    cloud_mask_optical,
    normalize_optical,
    compute_optical_features,
    calibrate_sar,
    apply_speckle_filter,
    apply_terrain_correction,
    normalize_sar,
    reproject_to_reference,
    align_modalities,
    validate_registration,
    fuse_early,
    OpticalSARModel,
    calculate_multimodal_confidence,
)

print("M3 Optical + SAR Module successfully loaded.")

## 2. Configuration

We instantiate the `OpticalSARConfig` configuration dataclass containing parameters for optical band mapping, normalization percentiles, SAR calibration, speckle filtering, registration resampling, and confidence weighting.

In [ ]:
config = OpticalSARConfig()

print("Master Pipeline Configuration:")
print(f"  Random Seed:             {config.random_seed}")
print(f"  Optical Normalization:   {config.optical.normalization_method} (percentiles: {config.optical.percentile_bounds})")
print(f"  SAR Speckle Filter:      {config.sar.speckle_filter_method} (kernel_size: {config.sar.speckle_kernel_size})")
print(f"  SAR Calibration to dB:   {config.sar.to_db} (range [{config.sar.min_db}, {config.sar.max_db}] dB)")
print(f"  Registration Resampling: {config.registration.resampling_method}")
print(f"  Confidence Weights:      {config.confidence.weights}")

## 3. Load Optical Satellite Image (Sentinel-2)

Expected data directory: `data/optical/`.
Loads multi-band Sentinel-2 GeoTIFF containing bands B02 (Blue), B03 (Green), B04 (Red), and B08 (NIR). If sample data is missing, a self-contained synthetic raster is created automatically.

In [ ]:
optical_path = project_root / "data" / "optical" / "sentinel2_real.tif"

# Fallback generator if real file is absent
if not optical_path.is_file():
    optical_path.parent.mkdir(parents=True, exist_ok=True)
    h, w = 256, 256
    y, x = np.mgrid[0:h, 0:w]
    river = np.exp(-((x - 80 - 20*np.sin(y/30.0))**2) / 50.0)
    veg = (np.sin(x/20.0) * np.cos(y/20.0) > 0.0).astype(np.float32)
    b2 = 400.0 - 200.0 * river + 50.0 * np.random.randn(h, w)
    b3 = 500.0 + 300.0 * veg - 250.0 * river
    b4 = 450.0 + 100.0 * veg - 300.0 * river
    b8 = 600.0 + 2000.0 * veg - 400.0 * river
    stack = np.stack([b2, b3, b4, b8], axis=0).astype(np.float32)
    t = affine.Affine(10.0, 0.0, 500000.0, 0.0, -10.0, 5200000.0)
    with rasterio.open(optical_path, 'w', driver='GTiff', height=h, width=w, count=4,
                       dtype='float32', crs=CRS.from_epsg(32632), transform=t, nodata=0.0) as dst:
        dst.write(stack)

optical_data = load_optical(optical_path, band_mapping={"B02": 1, "B03": 2, "B04": 3, "B08": 4})
print(f"Optical Loaded: shape={optical_data.shape}, CRS={optical_data.crs}, bands={optical_data.band_names}")
print(f"Resolution: {optical_data.resolution}, Valid Fraction: {optical_data.valid_fraction:.4f}")

## 4. Visualize Optical Imagery

Visualizing True Color (RGB: B04, B03, B02) and False Color Infrared (CIR: B08, B04, B03).

In [ ]:
from modules.optical_sar.optical.normalization import normalize_percentile

norm_vis = normalize_percentile(optical_data.data, lower_percentile=2.0, upper_percentile=98.0)
rgb_composite = np.clip(np.stack([norm_vis[2], norm_vis[1], norm_vis[0]], axis=-1), 0, 1) # B04, B03, B02
cir_composite = np.clip(np.stack([norm_vis[3], norm_vis[2], norm_vis[1]], axis=-1), 0, 1) # B08, B04, B03

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb_composite)
axes[0].set_title("Sentinel-2 True Color (RGB: B04, B03, B02)")
axes[0].axis("off")

axes[1].imshow(cir_composite)
axes[1].set_title("Sentinel-2 False Color CIR (B08, B04, B03)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. Optical Preprocessing

Applies external cloud/SCL masking if available (reporting explicit status when unavailable) and percentile normalization to $[0, 1]$.

In [ ]:
scl_path = project_root / "data" / "optical" / "sentinel2_scl.tif"
cloud_mask_arg = scl_path if scl_path.is_file() else None

cloud_res = cloud_mask_optical(optical_data, cloud_mask=cloud_mask_arg)
print(f"Cloud Masking Status: {cloud_res.status} (Cloud Fraction: {cloud_res.cloud_fraction})")

optical_norm = normalize_optical(cloud_res.optical_data, method="percentile", percentile_bounds=(1.0, 99.0))
print(f"Normalized Optical Data: shape={optical_norm.shape}, range=[{np.nanmin(optical_norm.data):.2f}, {np.nanmax(optical_norm.data):.2f}]")

## 6. Calculate NDVI & Spectral Indices

$$\text{NDVI} = \frac{\text{NIR} - \text{RED}}{\text{NIR} + \text{RED}}$$

In [ ]:
opt_features = compute_optical_features(optical_norm)
ndvi = opt_features["ndvi"]

valid_ndvi = ndvi[~np.isnan(ndvi)]
print(f"NDVI Statistics: Mean={np.mean(valid_ndvi):.4f}, Min={np.min(valid_ndvi):.4f}, Max={np.max(valid_ndvi):.4f}")

plt.figure(figsize=(7, 5))
im = plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
plt.colorbar(im, label="NDVI")
plt.title("Normalized Difference Vegetation Index (NDVI)")
plt.axis("off")
plt.show()

## 7. Load SAR Satellite Image (Sentinel-1)

Expected data directory: `data/sar/`.
Loads dual-polarization Sentinel-1 GeoTIFF with VV and VH channels.

In [ ]:
sar_path = project_root / "data" / "sar" / "sentinel1_real.tif"

if not sar_path.is_file():
    sar_path.parent.mkdir(parents=True, exist_ok=True)
    h, w = 256, 256
    vv = (np.random.exponential(scale=0.08, size=(h, w))).astype(np.float32)
    vh = (np.random.exponential(scale=0.02, size=(h, w))).astype(np.float32)
    sar_stack = np.stack([vv, vh], axis=0)
    t = affine.Affine(10.0, 0.0, 500000.0, 0.0, -10.0, 5200000.0)
    with rasterio.open(sar_path, 'w', driver='GTiff', height=h, width=w, count=2,
                       dtype='float32', crs=CRS.from_epsg(32632), transform=t, nodata=-9999.0) as dst:
        dst.write(sar_stack)

sar_data = load_sar(sar_path, polarizations=["VV", "VH"])
print(f"SAR Loaded: shape={sar_data.shape}, Polarizations={sar_data.polarizations}, CRS={sar_data.crs}")

## 8. Visualize SAR Polarizations

Displaying Sentinel-1 VV and VH backscatter channels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(sar_data.get_band("VV"), cmap="gray")
axes[0].set_title("Sentinel-1 VV Polarization")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(sar_data.get_band("VH"), cmap="gray")
axes[1].set_title("Sentinel-1 VH Polarization")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

## 9. SAR Preprocessing

1. Radiometric calibration (linear backscatter to dB: $\text{dB} = 10\log_{10}(\text{linear})$).
2. Vectorized Lee speckle filtering to reduce granular noise while preserving structural edges.
3. Terrain correction adapter status handling.
4. Percentile normalization.

In [ ]:
# 1. Calibration to dB
sar_cal = calibrate_sar(sar_data, is_already_calibrated=True, to_db=True, min_db=-35.0, max_db=5.0)

# 2. Lee Speckle Filter (k=5)
sar_filtered_arr = apply_speckle_filter(sar_cal.data, method="lee", kernel_size=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sar_cal.data[0], cmap="gray", vmin=-25, vmax=0)
axes[0].set_title("SAR VV (dB) — Speckled")
axes[0].axis("off")

axes[1].imshow(sar_filtered_arr[0], cmap="gray", vmin=-25, vmax=0)
axes[1].set_title("SAR VV (dB) — Lee Filtered (k=5)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# 3. Terrain correction check
sar_cal.data = sar_filtered_arr
terrain_res = apply_terrain_correction(sar_cal, dem_path=None)
print(f"Terrain Correction Status: {terrain_res.status} (Applied={terrain_res.applied})")

# 4. Normalization
sar_norm = normalize_sar(sar_cal, method="percentile")
print("SAR Preprocessing Complete.")

## 10. Geospatial Reprojection

Resamples and reprojects the SAR data onto the exact optical grid using `rasterio.warp.reproject`, matching CRS, resolution, bounds, and $(H, W)$ dimensions.

In [ ]:
sar_reprojected = reproject_to_reference(sar_norm, optical_norm, resampling_method="bilinear")

print(f"Optical Grid Dimensions: {optical_norm.shape[1:]} (CRS: {optical_norm.crs})")
print(f"Reprojected SAR Dimensions: {sar_reprojected.shape[1:]} (CRS: {sar_reprojected.crs})")
assert sar_reprojected.shape[1:] == optical_norm.shape[1:], "Dimensions must match exactly!"

## 11. Cross-Modal Fine Alignment

Estimates subpixel translation between optical and SAR structural gradient maps using Phase Correlation.

In [ ]:
alignment_res = align_modalities(optical_norm, sar_reprojected, method="phase_correlation")
print(f"Alignment Status: {alignment_res.status}")
print(f"Estimated Shift: dx={alignment_res.displacement[0]:.2f}px, dy={alignment_res.displacement[1]:.2f}px")
sar_aligned = alignment_res.aligned_sar

## 12. Registration Quality Validation

Computes quantitative registration metrics:
- Valid Overlap Ratio (IoU)
- Normalized Mutual Information (NMI)
- Structural Edge Gradient Correlation
- Composite Registration Score

In [ ]:
reg_val = validate_registration(optical_norm, sar_aligned)

print("=== Registration Quality Validation ===")
print(f"  Overlap Ratio:            {reg_val.overlap_ratio:.4f}")
print(f"  Normalized Mutual Info:   {reg_val.mutual_information:.4f}")
print(f"  Structural Edge Score:    {reg_val.structural_score:.4f}")
print(f"  Alignment Error:          {reg_val.alignment_error:.2f} px")
print(f"  Composite Score:          {reg_val.registration_score:.4f}")
print(f"  Validation Passed:        {reg_val.passed}")

## 13. Early Multimodal Fusion

Concatenates registered Optical bands `[B02, B03, B04, B08]` and SAR polarizations `[VV, VH]` into a 6-channel composite.

In [ ]:
early_fusion = fuse_early(optical_norm, sar_aligned)
print(f"Fused Tensor Shape: {early_fusion.shape}")
print(f"Channel Names:      {early_fusion.channel_names}")

## 14. Multimodal Deep Learning Model Inference

Demonstrating inference using the two-stream PyTorch `OpticalSARModel` (`OpticalEncoder` + `SAREncoder` + `MultimodalFusionModule` + `ClassificationHead`).

In [ ]:
model = OpticalSARModel(
    fusion_type="feature",
    optical_channels=optical_norm.channels,
    sar_channels=sar_aligned.channels,
    feature_dim=128,
    num_classes=5
)

inference_result = model.predict(optical_norm.data, sar_aligned.data)

print("Model Inference Result:")
print(f"  Predicted Class Index: {inference_result.predicted_class}")
print(f"  Probabilities:         {[round(p, 4) for p in inference_result.probabilities]}")
print(f"  Model Confidence:      {inference_result.model_confidence:.4f}")
print(f"  Status:                {inference_result.status}")
print(f"  Scientific Notes:      {inference_result.notes}")

## 15. Multimodal Confidence Estimation

Combines data validity, cloud cover, registration score, and model confidence into an operational confidence score.

In [ ]:
conf_assessment = calculate_multimodal_confidence(
    optical_data=optical_norm,
    sar_data=sar_aligned,
    registration_result=reg_val,
    model_result=inference_result,
    weights=config.confidence.weights,
    thresholds=config.confidence.thresholds,
)

print("=== Multimodal Confidence Assessment ===")
print(f"  Confidence Score: {conf_assessment.score:.4f} ({conf_assessment.level.upper()})")
print("  Component Breakdown:")
for comp, val in conf_assessment.components.items():
    print(f"    - {comp:22s}: {val:.4f}")
print(f"  Notes: {conf_assessment.notes}")

## 16. Final Multimodal Visualizations

Side-by-side display of Optical True Color (RGB), NDVI, Registered SAR VV backscatter, and Fused Multimodal False Color Composite.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Optical RGB
axes[0].imshow(rgb_composite)
axes[0].set_title("1. Sentinel-2 (True Color RGB)")
axes[0].axis("off")

# 2. Optical NDVI
im_ndvi = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
axes[1].set_title("2. Optical NDVI")
axes[1].axis("off")
plt.colorbar(im_ndvi, ax=axes[1], fraction=0.046)

# 3. Registered SAR VV
im_sar = axes[2].imshow(sar_aligned.get_band("VV"), cmap="gray")
axes[2].set_title("3. Registered SAR VV")
axes[2].axis("off")
plt.colorbar(im_sar, ax=axes[2], fraction=0.046)

# 4. Fused Multimodal Composite (R: NIR, G: Red, B: SAR VV)
mm_comp = np.clip(np.stack([
    optical_norm.get_band("B08"),
    optical_norm.get_band("B04"),
    sar_aligned.get_band("VV")
], axis=-1), 0, 1)
axes[3].imshow(mm_comp)
axes[3].set_title("4. Fused Multimodal Composite (NIR, Red, SAR)")
axes[3].axis("off")

plt.suptitle(f"SatQuery M3 Multimodal Analysis — Reg Score: {reg_val.registration_score:.2f} | Confidence: {conf_assessment.score:.2f} ({conf_assessment.level.upper()})", fontsize=14)
plt.tight_layout()
plt.show()

## 17. Summary & Module Handoff Interfaces

The M3 module establishes clean, decoupled handoff interfaces for integration with the other Sat Query modules:

| Module | Role | Data Consumed from M3 | Interface Method |
| :--- | :--- | :--- | :--- |
| **M2** | Change Detection & Grounding | Registered Optical/SAR rasters, NDVI, early-fused tensor | `result.optical_data`, `result.registered_sar_data`, `result.early_fusion_result.fused_data` |
| **M4** | Agent Orchestration | Structured analysis JSON, prediction, confidence tier | `result.to_dict()` |
| **M5** | GIS & Evidence | CRS, bounds, pixel resolution, affine geotransform | `result.to_gis_evidence()` |
| **M6** | App & API Integration | High-level serializable dictionary, GeoTIFF exports | `result.to_dict()` |

In [ ]:
# Execute full master pipeline in a single call
pipeline_out = run_optical_sar_pipeline(
    optical_path=optical_path,
    sar_path=sar_path,
    config=config,
    run_inference=False,
)

print("Master Pipeline Result Status:", pipeline_out.status)
print("M4 JSON Payload Keys:        ", list(pipeline_out.to_dict().keys()))
print("M5 GIS Evidence Keys:        ", list(pipeline_out.to_gis_evidence().keys()))